# Modular semi-Mapper pipeline — CTN-0051

This notebook is a thin driver over the `mapper` package. All logic lives in the package modules; here you only **tune parameters and call the pipeline**.

Pipeline: `data -> distance -> lens -> cover -> graph -> layout -> viz`.

Three lenses are available:
- **feature**    — a data column (original behaviour, e.g. `attendance_density_w8`)
- **centrality** — graph centrality on the proximity graph (degree, betweenness, ...)
- **density**    — local density in feature space (knn, ball, kde)

## 1. Parameters — tune here

In [9]:
from mapper import MapperParams, run_pipeline, visualise
from mapper import diagnostics as dg
import numpy as np

params = MapperParams(
    PRE_TRIAL_CSV="../../data/clean-data/pre-trial.csv",
    TARGET_CSV   ="../../data/clean-data/retention_tier.csv",
    W8_CSV       ="../../data/clean-data/features_w8.csv",
    W12_CSV      ="../../data/clean-data/features_w12.csv",

    # --- proximity ---
    EPSILON=2.15,
    METRIC ="minkowski",        # euclidean | cosine | manhattan | minkowski
    MINKOWSKI_P= np.inf,            # only for minkowski

    # --- LENS: pick one of the three ---
    LENS_KIND="feature",        # "feature" | "centrality" | "density"
    FEATURE_LENS_COL  ="engagement_onset_w12",   # feature lens: attendance_density_w8 | detox_los 
    CENTRALITY_MEASURE="betweenness",             # centrality lens
    DENSITY_METHOD    ="knn",                     # density lens
    DENSITY_K         =10,

    # --- COVER / BINNING (tunable) ---
    COVER_MODE ="piecewise",      # "uniform" (N_INTERVALS+OVERLAP) or "edges" (BIN_EDGES)
    N_INTERVALS= 8,
    OVERLAP    =0.5,
    # For explicit clinical bins instead, use:
    # COVER_MODE="edges", BIN_EDGES=[0,3,7,14,21], BIN_LABELS=["0-3","3-7","7-14","14+"]

    PIECEWISE_SEGMENTS=[
        (0.0, 0.3, 5),    # dense cover in the first half
        (0.3, 0.6, 100),
        (0.6, 1.0, 5)   # sparse cover in the second half
    ],


    # --- edge rule for the displayed graph ---
    EDGE_RULE  ="cover",        # "cover" (share a set) | "gap" | "none"
    MAX_BIN_GAP=1,

    # --- layout & colour ---
    LAYOUT  ="spring", 
    SPRING_K = 100,              # spring | spectral | pca
    COLOR_BY="tier",            # "tier" or "lens"
)
print(params.summary())

ε=2.15 | metric=minkowski(p=inf) | lens=feature=engagement_onset_w12 | cover=edges | edge_rule=cover | layout=spring


## 2. Run the pipeline

In [10]:
result = run_pipeline(params)   # prints stage-by-stage summaries

Patients        : 554
Feature matrix  : (554, 18)
Missing values  : 0

Retention tier distribution:
retention_tier
1    164
2    108
3     61
4    221 

Distance matrix : (554, 554)
Distance range  : [0.000, 17.817]
Percentiles:
   10th : 2.151
   25th : 2.458
   50th : 2.592
   75th : 3.528
   90th : 4.750
With ε = 2.15:
  Edges before pruning : 14553
  Edge density         : 9.5% 

[feature] engagement_onset_w12: min=0.000 median=0.000 max=2.000 (n_valid=554) 

Cover mode: piecewise  |  110 sets

  Set  0 [0,0.12)        : 436 patients
  Set  1 [0.06,0.18)     :   0 patients  <- empty
  Set  2 [0.12,0.24)     :   0 patients  <- empty
  Set  3 [0.18,0.3)      :   0 patients  <- empty
  Set  4 [0.24,0.36)     :   0 patients  <- empty
  Set  5 [0.3,0.306)     :   0 patients  <- empty
  Set  6 [0.303,0.309)   :   0 patients  <- empty
  Set  7 [0.306,0.312)   :   0 patients  <- empty
  Set  8 [0.309,0.315)   :   0 patients  <- empty
  Set  9 [0.312,0.318)   :   0 patients  <- empty
  Set 

## 3. Interactive Bokeh graph

In [11]:
from bokeh.io import output_notebook
output_notebook()
visualise(result)

Loading BokehJS ...

figure(id='p1210', ...)

## Graph Grid

### Grid setup — parameter space

| Axis | Values |
|------|--------|
| **Lens** | `feature=attendance_density_w8`, `feature=detox_los`, `density-knn k=5` |
| **Metric / ε** | cosine ε∈{0.5,0.6,0.7,0.8} · mink p=2 ε∈{4.5,5.5,6.5} · mink p=4 ε∈{3,3.5,4} · mink p=∞ ε∈{2.4,2.5,2.6,3} |
| **Overlap** | 0.5, 0.7 |

All other params fixed: `N_intervals=80`, `edge_rule=cover`, `max_bin_gap=1`, `spring_k=10`.  
Output: `w8/` — 84 individual PNGs + one composite grid.

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from mapper import MapperParams, MapperResult, TIER_COLORS, TIER_LABELS
import mapper.data     as _data_mod
import mapper.distance as _dist_mod
import mapper.lenses   as _lens_mod
import mapper.cover    as _cover_mod
import mapper.graph    as _graph_mod
import mapper.layout   as _layout_mod

OUT_DIR = "w8"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Parameter grid ────────────────────────────────────────────────────── #
LENS_CONFIGS = [
    dict(label="feature_att_w8",    LENS_KIND="feature",  FEATURE_LENS_COL="attendance_density_w8", DENSITY_K=5),
    dict(label="feature_detox_los", LENS_KIND="feature",  FEATURE_LENS_COL="detox_los",              DENSITY_K=5),
    dict(label="density_knn5",      LENS_KIND="density",  FEATURE_LENS_COL="attendance_density_w8", DENSITY_K=5),
]

METRIC_CONFIGS = [
    dict(label="cosine_e0.5",      display="cosine  ε=0.5",      METRIC="cosine",    MINKOWSKI_P=2,      EPSILON=0.5),
    dict(label="cosine_e0.6",      display="cosine  ε=0.6",      METRIC="cosine",    MINKOWSKI_P=2,      EPSILON=0.6),
    dict(label="cosine_e0.7",      display="cosine  ε=0.7",      METRIC="cosine",    MINKOWSKI_P=2,      EPSILON=0.7),
    dict(label="cosine_e0.8",      display="cosine  ε=0.8",      METRIC="cosine",    MINKOWSKI_P=2,      EPSILON=0.8),
    dict(label="mink_p2_e4.5",     display="mink p=2  ε=4.5",    METRIC="minkowski", MINKOWSKI_P=2,      EPSILON=4.5),
    dict(label="mink_p2_e5.5",     display="mink p=2  ε=5.5",    METRIC="minkowski", MINKOWSKI_P=2,      EPSILON=5.5),
    dict(label="mink_p2_e6.5",     display="mink p=2  ε=6.5",    METRIC="minkowski", MINKOWSKI_P=2,      EPSILON=6.5),
    dict(label="mink_p4_e3.0",     display="mink p=4  ε=3.0",    METRIC="minkowski", MINKOWSKI_P=4,      EPSILON=3.0),
    dict(label="mink_p4_e3.5",     display="mink p=4  ε=3.5",    METRIC="minkowski", MINKOWSKI_P=4,      EPSILON=3.5),
    dict(label="mink_p4_e4.0",     display="mink p=4  ε=4.0",    METRIC="minkowski", MINKOWSKI_P=4,      EPSILON=4.0),
    dict(label="mink_pinf_e2.4",   display="mink p=∞  ε=2.4",    METRIC="minkowski", MINKOWSKI_P=np.inf, EPSILON=2.4),
    dict(label="mink_pinf_e2.5",   display="mink p=∞  ε=2.5",    METRIC="minkowski", MINKOWSKI_P=np.inf, EPSILON=2.5),
    dict(label="mink_pinf_e2.6",   display="mink p=∞  ε=2.6",    METRIC="minkowski", MINKOWSKI_P=np.inf, EPSILON=2.6),
    dict(label="mink_pinf_e3.0",   display="mink p=∞  ε=3.0",    METRIC="minkowski", MINKOWSKI_P=np.inf, EPSILON=3.0),
]

OVERLAPS = [0.5, 0.7]

BASE_KW = dict(
    PRE_TRIAL_CSV="../../data/clean-data/pre-trial.csv",
    TARGET_CSV   ="../../data/clean-data/retention_tier.csv",
    W8_CSV       ="../../data/clean-data/features_w8.csv",
    COVER_MODE   ="uniform",
    N_INTERVALS  =80,
    EDGE_RULE    ="cover",
    MAX_BIN_GAP  =1,
    LAYOUT       ="spring",
    SPRING_K     =10,
    COLOR_BY     ="tier",
)

# ── Display helpers ───────────────────────────────────────────────────── #
_TIER_PATCHES = [
    mpatches.Patch(color=TIER_COLORS[t], label=f"T{t}: {TIER_LABELS[t]}")
    for t in [1, 2, 3, 4]
]

def _lens_str(lc):
    if lc["LENS_KIND"] == "feature":
        col = "att_w8" if "attendance" in lc["FEATURE_LENS_COL"] else "detox_los"
        return f"feature={col}"
    return f"density-knn  k={lc['DENSITY_K']}"

def _draw(ax, G, title, node_size=5, fontsize=7):
    pos = {n: (G.nodes[n]["x"], G.nodes[n]["y"]) for n in G.nodes}
    colors = [TIER_COLORS.get(G.nodes[n]["tier"], "#aaaaaa") for n in G.nodes]
    n_nodes  = G.number_of_nodes()
    n_edges  = G.number_of_edges()
    n_cc     = nx.number_connected_components(G)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.12, width=0.25, edge_color="#999999")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors,
                           node_size=node_size, alpha=0.85, linewidths=0.2)
    ax.set_title(f"{title}\nn={n_nodes}  e={n_edges}  cc={n_cc}", fontsize=fontsize, pad=2)
    ax.axis("off")

total_combos = len(LENS_CONFIGS) * len(METRIC_CONFIGS) * len(OVERLAPS)
print(f"Grid: {len(LENS_CONFIGS)} lenses  x  {len(METRIC_CONFIGS)} metric/ε  x  {len(OVERLAPS)} overlaps  =  {total_combos} graphs")
print(f"Output directory: {os.path.abspath(OUT_DIR)}")

Grid: 3 lenses  x  14 metric/ε  x  2 overlaps  =  84 graphs
Output directory: c:\Users\Miki\OneDrive - Universitat de Barcelona\MÀSTER\TFM\REPO\Masters-Final-Project\Algorithms\mapper_pipeline\w8


In [5]:
# ── 1. Load dataset once ─────────────────────────────────────────────── #
print("Loading dataset...")
_seed_p = MapperParams(
    **BASE_KW,
    LENS_KIND="feature", FEATURE_LENS_COL="attendance_density_w8",
    METRIC="cosine", EPSILON=0.5, OVERLAP=0.5, DENSITY_K=5,
)
_ds = _data_mod.load_dataset(_seed_p)
print(f"  {_ds.X.shape[0]} patients  x  {_ds.X.shape[1]} features\n")

# ── 2. Distance matrices — one per unique (metric, p, ε) ─────────────── #
print("Computing distance matrices (14 unique configs)...")
_dist_cache = {}
for mc in METRIC_CONFIGS:
    _key = (mc["METRIC"], mc["MINKOWSKI_P"], mc["EPSILON"])
    if _key not in _dist_cache:
        _p = MapperParams(
            **BASE_KW,
            METRIC=mc["METRIC"], MINKOWSKI_P=mc["MINKOWSKI_P"], EPSILON=mc["EPSILON"],
            LENS_KIND="feature", FEATURE_LENS_COL="attendance_density_w8",
            OVERLAP=0.5, DENSITY_K=5,
        )
        print(f"  {mc['display']}...", end=" ", flush=True)
        _dist_cache[_key] = _dist_mod.compute_distances(_ds.X, _p)
        print("done")
print()

# ── 3. Run full grid, save individual images ─────────────────────────── #
_results = {}   # (lens_label, metric_label, overlap) -> MapperResult | None
_total = len(LENS_CONFIGS) * len(METRIC_CONFIGS) * len(OVERLAPS)
_done  = 0

for lc in LENS_CONFIGS:
    for mc in METRIC_CONFIGS:
        _D = _dist_cache[(mc["METRIC"], mc["MINKOWSKI_P"], mc["EPSILON"])]
        for ov in OVERLAPS:
            _p = MapperParams(
                **BASE_KW,
                LENS_KIND        = lc["LENS_KIND"],
                FEATURE_LENS_COL = lc["FEATURE_LENS_COL"],
                DENSITY_K        = lc["DENSITY_K"],
                METRIC           = mc["METRIC"],
                MINKOWSKI_P      = mc["MINKOWSKI_P"],
                EPSILON          = mc["EPSILON"],
                OVERLAP          = ov,
            )
            _key = (lc["label"], mc["label"], ov)
            _done += 1
            try:
                _lens  = _lens_mod.build_lens(_p, _ds.df, _D)
                _cover = _cover_mod.build_cover(_lens.values, _p)
                _G     = _graph_mod.build_epsilon_graph(
                             _D, _ds.df, cover=_cover, lens=_lens,
                             edge_rule=_p.EDGE_RULE, epsilon=_p.EPSILON,
                             max_bin_gap=_p.MAX_BIN_GAP)
                _layout_mod.compute_layout(_G, _ds.X, _p)
                _r = MapperResult(dataset=_ds, D=_D, lens=_lens,
                                  cover=_cover, graph=_G, params=_p)
                _results[_key] = _r

                # ── individual image ─────────────────────────────────── #
                _fig, _ax = plt.subplots(figsize=(5, 4.8))
                _title = (f"Lens:    {_lens_str(lc)}\n"
                          f"Metric:  {mc['display']}\n"
                          f"Overlap: {ov}   N_intervals: 80   edge_rule: cover")
                _draw(_ax, _G, _title, node_size=8, fontsize=9)
                _ax.legend(handles=_TIER_PATCHES, loc="lower right",
                           fontsize=6.5, framealpha=0.85, markerscale=0.8)
                _fname = f"{lc['label']}__{mc['label']}__ov{ov}.png"
                _fig.savefig(os.path.join(OUT_DIR, _fname), dpi=120, bbox_inches="tight")
                plt.close(_fig)
                print(f"[{_done:2d}/{_total}] {_fname}")

            except Exception as _exc:
                _results[_key] = None
                print(f"[{_done:2d}/{_total}] ERROR  {lc['label']}  {mc['label']}  ov={ov}: {_exc}")

_ok = sum(v is not None for v in _results.values())
print(f"\nCompleted: {_ok}/{_total} successful — images in '{OUT_DIR}/'")

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '../data/clean-data/features_w12.csv'

In [ ]:
# Composite grid image
# Layout: rows = lens x overlap  (6 rows),  cols = metric+epsilon  (14 cols)

_N_ROWS = len(LENS_CONFIGS) * len(OVERLAPS)   # 6
_N_COLS = len(METRIC_CONFIGS)                  # 14
_CW, _RH = 2.6, 2.8

_fig, _axes = plt.subplots(
    _N_ROWS, _N_COLS,
    figsize=(_N_COLS * _CW, _N_ROWS * _RH),
    gridspec_kw={"hspace": 0.55, "wspace": 0.08},
)

_fig.suptitle(
    "Mapper graph grid  |  N_intervals=80  |  edge_rule=cover  |  spring_k=10"
    "Rows: lens x overlap     Columns: metric / epsilon",
    fontsize=11, fontweight="bold", y=1.01,
)

for ci, mc in enumerate(METRIC_CONFIGS):
    _axes[0, ci].set_xlabel(mc["display"], fontsize=6.5, labelpad=2)
    _axes[0, ci].xaxis.set_label_position("top")

_ri = 0
for lc in LENS_CONFIGS:
    for ov in OVERLAPS:
        _axes[_ri, 0].set_ylabel(
            f"{_lens_str(lc)}\nov={ov}", fontsize=6.5,
            rotation=90, labelpad=4, ha="right", va="center",
        )
        _ri += 1

_ri = 0
for lc in LENS_CONFIGS:
    for ov in OVERLAPS:
        for ci, mc in enumerate(METRIC_CONFIGS):
            _ax  = _axes[_ri, ci]
            _key = (lc["label"], mc["label"], ov)
            _r   = _results.get(_key)
            if _r is None:
                _ax.text(0.5, 0.5, "ERR", ha="center", va="center",
                         color="red", fontsize=8, transform=_ax.transAxes)
                _ax.axis("off")
            else:
                _compact = f"{_lens_str(lc)}\n{mc['display']}  ov={ov}"
                _draw(_ax, _r.graph, _compact, node_size=3, fontsize=5.5)
        _ri += 1

_fig.legend(
    handles=_TIER_PATCHES, loc="lower center", ncol=4,
    fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.02),
)

_grid_path = os.path.join(OUT_DIR, "grid__all.png")
_fig.savefig(_grid_path, dpi=100, bbox_inches="tight")
plt.close(_fig)
print(f"Composite grid saved -> {_grid_path}")
print(f"  ({_N_ROWS} rows x {_N_COLS} cols  =  {_N_ROWS * _N_COLS} panels)")


Composite grid saved -> w8\grid__all.png
  (6 rows x 14 cols  =  84 panels)
